# queue-boarding/v1.0.0 — 학습·추론 독립 노트북

이 노트북은 프로젝트의 탑승 가능성 주 모델을 한 곳에서 재현하고 사용하는 진입점입니다.

> **데이터 정책:** 2026-08-18은 봉인된 최종 평가 데이터입니다. 재학습·보정·모델 선택에 사용하지 않습니다. 등록 artifact 재생성은 2026-08-13까지의 고정 split만 사용합니다.


In [ ]:
from pathlib import Path
import json, sys
import joblib
import pandas as pd

REPO = Path.cwd()
if not (REPO / 'analysis').is_dir():
    REPO = Path('/content/10th-toy-team4')
sys.path.insert(0, str(REPO / 'analysis'))
sys.path.insert(0, str(REPO))

from queue_boarding_v1_inference import QueueBoardingV1
from queue_boarding_v1_training import train_bundle

REGISTRY = REPO / 'analysis/model_registry/queue-boarding/registry.json'
ARTIFACT = REPO / 'analysis/model_registry/queue-boarding/v1.0.0/queue_boarding_v1_0_0.pkl'
print(REGISTRY)
print(ARTIFACT)


## 1. 고정 계약 확인

- 입력: 동시에 보이는 다음 버스 3대의 도착 전 prepared snapshot
- 사용자 입력: 앞 대기 인원 `queue_ahead`
- 좌석 분포: state-profile 점 예측 25% + hybrid PMF 75%
- 출력: 1·2·3번째 버스까지 누적 탑승확률과 `[0,1,2,3+]` 통과 대수 클래스


In [ ]:
registry = json.loads(REGISTRY.read_text(encoding='utf-8'))
metadata = json.loads(ARTIFACT.with_suffix('.metadata.json').read_text(encoding='utf-8'))
display({
    'primary_version': registry['primary_version'],
    'model_id': metadata['model_id'],
    'source_cutoff': metadata['source_cutoff'],
    'source_fingerprint': metadata['source_fingerprint'],
    'artifact': metadata['artifact'],
    'boarding_policy': metadata['boarding_policy'],
})


## 2. 선택적 재학습

기본값은 `False`입니다. 재학습 시에도 등록 artifact를 덮어쓰지 않고 별도 파일에 저장합니다. `train_bundle()` 내부에서 동결 cache 검증과 날짜 제한을 강제합니다.


In [ ]:
RETRAIN = False
if RETRAIN:
    rebuilt = train_bundle()
    rebuild_path = REPO / 'artifacts/queue_boarding_v1_0_0_rebuild.pkl'
    rebuild_path.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(rebuilt, rebuild_path, compress=3)
    print(rebuild_path)


## 3. Primary artifact 로드

로더는 registry의 경로와 SHA-256을 검증한 뒤에만 역직렬화합니다.


In [ ]:
model = QueueBoardingV1.load(ARTIFACT, registry_path=REGISTRY)
print(model.bundle['model_id'])
print('required prepared features:', len(model.bundle['required_prepared_input_features']))


## 4. 추론 입력

한 질의는 정확히 3행이며 `bus_order=1,2,3`이어야 합니다. 같은 `query_id`의 `queue_ahead`는 동일해야 합니다. 예제는 최종 평가일의 **라벨을 제거한 입력 3행**이며 추론 검증에만 사용합니다.


In [ ]:
example_path = ARTIFACT.parent / 'example_input.csv'
example = pd.read_csv(example_path)
display(example[['query_id', 'bus_order', 'queue_ahead', 'route_id', 'snapshot_remaining_seats', 'target_stop_gap']])


In [ ]:
prediction = model.predict_queries(example)
display(prediction.T)

probability_columns = [f'board_by_{index}_probability' for index in (1, 2, 3)]
assert prediction[probability_columns].notna().all().all()
assert prediction[probability_columns].apply(lambda column: column.between(0, 1).all()).all()
assert prediction.loc[0, probability_columns].is_monotonic_increasing


## 5. 배치 추론

여러 `query_id`를 이어 붙인 CSV도 동일하게 처리합니다.

```bash
PYTHONPATH=analysis python analysis/queue_boarding_v1_inference.py \
  --input-csv prepared_queries.csv \
  --output-csv boarding_predictions.csv
```


In [ ]:
# 사용자 파일로 교체하세요.
INPUT_CSV = example_path
batch = pd.read_csv(INPUT_CSV)
batch_prediction = model.predict_queries(batch)
display(batch_prediction)


## 6. 최종 테스트 결과(읽기 전용)

이 셀은 보고만 하며 어떤 학습 함수에도 결과를 전달하지 않습니다.


In [ ]:
summary_path = REPO / 'analysis/final_evaluation_2026-08-18/results/summary.json'
if summary_path.is_file():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    display(summary['queue_boarding_model'])
else:
    print('Final-evaluation summary is not present in this checkout.')


## 해석 원칙

1. `board_by_1_probability`는 첫 버스 탑승확률입니다.
2. `predicted_sent_class=3`은 정확히 3대가 아니라 **3대 안에 못 탐/3대 이상**입니다.
3. 현재 모델은 FIFO, 질의 후 신규 승객 없음, 버스별 좌석분포 독립을 가정합니다.
4. 실제 현장 대기열 검증 전에는 확률을 안내하되 통과 대수를 확정적으로 보장하지 않습니다.
